In [1]:
import krippendorff
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
from itertools import combinations
from sklearn.metrics import cohen_kappa_score
from pathlib import Path
import pandas as pd

# Read Labelled Veteran Data (Excel)

This notebook loads `Labelled-consolidatedVeteranData-May.xlsx` and previews each sheet.

In [2]:
DATA_PATH = Path("Labelled-consolidatedVeteranData-May.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected Excel file at {DATA_PATH.resolve()}")

In [3]:
xl = pd.ExcelFile(DATA_PATH)
print("Sheets:", xl.sheet_names)

Sheets: ['Form Responses 1', 'Sheet1', 'First group']


In [4]:
sheets = {name: pd.read_excel(xl, sheet_name=name) for name in xl.sheet_names}

for name, df in sheets.items():
    print(f"\n{name}: {df.shape[0]} rows x {df.shape[1]} columns")


Form Responses 1: 9 rows x 1922 columns

Sheet1: 5567 rows x 11 columns

First group: 149 rows x 14 columns


## Form Responses 1

Wide form-response export (one column per survey question).

In [5]:
form_responses = sheets["Form Responses 1"]
print(f"Shape: {form_responses.shape}")
form_responses.iloc[:, :5].head()

Shape: (9, 1922)


,Timestamp,Email Address,Score,Love me? 11,Happy birthday to the beloved king of grunge #KurtCobain1
0,2023-05-25 07:50:21.880,nadiyah@jetconstellations.com,0,NaN,NaN
1,2024-02-13 11:46:13.943,pborucki@mcw.edu,0,NaN,1-No Concern
2,2024-02-19 07:34:35.863,pglowinski@mcw.edu,0,NaN,1-No Concern
3,2024-03-05 15:06:45.606,ddrive345@gmail.com,0,NaN,6- Not Relevant to the Research
4,2024-03-05 15:24:43.760,scottoldwolf@aol.com,0,NaN,1-No Concern


## Reformat Form Responses 1

Pivot the wide Google Form export so that:

- **First column** = tweet text (~149 rows)
- **Column headers** = 8 reviewer email addresses (Excel rows 3–10)
- **Values** = each reviewer's label for that tweet

In [6]:
META_COLS = ["Timestamp", "Email Address", "Score"]

# Excel rows 3-10 (0-based iloc 1:9) are the 8 reviewers
reviewers = form_responses.iloc[1:9].copy()

tweet_cols = [c for c in form_responses.columns if c not in META_COLS]
labeled_tweet_cols = [c for c in tweet_cols if reviewers[c].notna().any()]

print(f"Reviewers: {len(reviewers)}")
print(f"Reviewed tweets: {len(labeled_tweet_cols)}")

reviews_by_tweet = (
    reviewers.set_index("Email Address")[labeled_tweet_cols]
    .T
    .reset_index(names="tweet")
)

reviews_by_tweet.shape

Reviewers: 8
Reviewed tweets: 149


(149, 9)

In [7]:
# First column = tweet, remaining columns = reviewer labels
reviews_by_tweet.head()

Email Address,tweet,pborucki@mcw.edu,pglowinski@mcw.edu,ddrive345@gmail.com,scottoldwolf@aol.com,michaelschmoker@gmail.com,lorih39@yahoo.com,gregwilliams1228@gmail.com,johnlacheson@gmail.com
0,Happy birthday to the beloved king of grunge #...,1-No Concern,1-No Concern,6- Not Relevant to the Research,1-No Concern,2- Distressed Self-Disclosure,1-No Concern,1-No Concern,6- Not Relevant to the Research
1,Amy lees plushy game is on point,1-No Concern,1-No Concern,1-No Concern,Other,1-No Concern,1-No Concern,6- Not Relevant to the Research,2- Distressed Self-Disclosure
2,Rush Limbaugh has died rot in pieces you racis...,2- Distressed Self-Disclosure,1-No Concern,Other,1-No Concern,3-Self-Identified Crisis Symptoms and Seeking ...,2- Distressed Self-Disclosure,1-No Concern,1-No Concern
3,Round&round electricity goes where it stop PLE...,2- Distressed Self-Disclosure,1-No Concern,1-No Concern,4- Self-Identified Crisis Symptoms but Not See...,2- Distressed Self-Disclosure,1-No Concern,2- Distressed Self-Disclosure,6- Not Relevant to the Research
4,Dear winter weather GET THE FUCK OUT OF TEXAS ...,1-No Concern,2- Distressed Self-Disclosure,6- Not Relevant to the Research,1-No Concern,6- Not Relevant to the Research,2- Distressed Self-Disclosure,3-Self-Identified Crisis Symptoms and Seeking ...,2- Distressed Self-Disclosure


## Anonymize reviewer emails

Map each reviewer email to `reviewer1`–`reviewer8` and export a shareable dataset.

In [8]:
reviewer_emails = [c for c in reviews_by_tweet.columns if c != "tweet"]
reviewer_map = {email: f"reviewer{i + 1}" for i, email in enumerate(reviewer_emails)}

reviewer_id_map = pd.DataFrame(
    {"reviewer_id": reviewer_map.values(), "email": reviewer_map.keys()}
)

veteran_tweet_labels = reviews_by_tweet.rename(columns=reviewer_map)

ANON_DATA_PATH = Path("Veteran_Tweet_Label_PTSD.csv")
MAP_PATH = Path("reviewer_id_map.csv")

veteran_tweet_labels.to_csv(ANON_DATA_PATH, index=False)
reviewer_id_map.to_csv(MAP_PATH, index=False)

print(f"Saved anonymized data to {ANON_DATA_PATH.resolve()}")
print(f"Saved reviewer mapping to {MAP_PATH.resolve()}")
reviewer_id_map

Saved anonymized data to \\venn.mscsnet.mu.edu\accounts\kislam\Documents\Projects\LLM-Data-Annotation\Veteran_Tweet_Label_PTSD.csv
Saved reviewer mapping to \\venn.mscsnet.mu.edu\accounts\kislam\Documents\Projects\LLM-Data-Annotation\reviewer_id_map.csv


,reviewer_id,email
0,reviewer1,pborucki@mcw.edu
1,reviewer2,pglowinski@mcw.edu
2,reviewer3,ddrive345@gmail.com
3,reviewer4,scottoldwolf@aol.com
4,reviewer5,michaelschmoker@gmail.com
5,reviewer6,lorih39@yahoo.com
6,reviewer7,gregwilliams1228@gmail.com
7,reviewer8,johnlacheson@gmail.com


In [9]:
OUTPUT_PATH = Path("form_responses_reformatted.csv")
reviews_by_tweet.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH.resolve()}")

Saved to \\venn.mscsnet.mu.edu\accounts\kislam\Documents\Projects\LLM-Data-Annotation\form_responses_reformatted.csv


In [10]:
import pandas as pd
from collections import Counter

# ============================================================
# 1. READ ORIGINAL DATA
# ============================================================

df = pd.read_csv("Veteran_Tweet_Label_PTSD.csv")

# ============================================================
# 2. IDENTIFY REVIEWER COLUMNS
# ============================================================

reviewer_cols = [
    c for c in df.columns
    if c.startswith("reviewer")
]

print(f"Number of reviewers: {len(reviewer_cols)}")
print(f"Number of tweets: {len(df)}")


# ============================================================
# 3. SEVERITY ORDER
#
# Higher number = more severe
#
# Only substantive research labels are included here.
# "Other" and "Not Relevant" are not considered severity levels.
# ============================================================

SEVERITY_ORDER = {
    "1-No Concern": 1,
    "2- Distressed Self-Disclosure": 2,
    "3-Self-Identified Crisis Symptoms and Seeking Support": 3,
    "4- Self-Identified Crisis Symptoms but Not Seeking Support": 4,
    "5- Acute Crisis with Suicidal Ideation": 5,
}


# ============================================================
# 4. MAJORITY VOTE WITH SEVERITY-BASED TIE BREAKING
# ============================================================

def majority_vote_severity(row):

    labels = row.dropna().tolist()

    if len(labels) == 0:
        return None

    counts = Counter(labels)

    # Highest number of votes
    max_count = max(counts.values())

    # Labels receiving the highest number of votes
    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # --------------------------------------------------------
    # Clear majority
    # --------------------------------------------------------

    if len(winners) == 1:
        return winners[0]

    # --------------------------------------------------------
    # Tie
    #
    # Among tied substantive labels, select the most severe.
    # --------------------------------------------------------

    severity_winners = [
        label
        for label in winners
        if label in SEVERITY_ORDER
    ]

    if severity_winners:

        return max(
            severity_winners,
            key=lambda label: SEVERITY_ORDER[label]
        )

    # --------------------------------------------------------
    # If all tied labels are non-substantive categories
    # such as Other / Not Relevant, keep the tie unresolved.
    # --------------------------------------------------------

    return " | ".join(sorted(winners))


# ============================================================
# 5. NUMBER OF VOTES FOR SELECTED GROUND-TRUTH LABEL
# ============================================================

def majority_vote_count(row):

    labels = row.dropna().tolist()

    if len(labels) == 0:
        return 0

    counts = Counter(labels)

    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # Clear majority
    if len(winners) == 1:
        return max_count

    # Tie resolved using severity
    severity_winners = [
        label
        for label in winners
        if label in SEVERITY_ORDER
    ]

    if severity_winners:

        selected_label = max(
            severity_winners,
            key=lambda label: SEVERITY_ORDER[label]
        )

        return counts[selected_label]

    # Unresolved non-substantive tie
    return 0


# ============================================================
# 6. IDENTIFY ORIGINAL TIE CASES
#
# This checks ties BEFORE severity-based resolution.
# ============================================================

def get_tie_labels(row):

    labels = row.dropna().tolist()

    if len(labels) == 0:
        return None

    counts = Counter(labels)

    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # No tie
    if len(winners) == 1:
        return None

    # Preserve all tied labels and their vote counts
    return " | ".join(
        f"{label} ({counts[label]} votes)"
        for label in sorted(winners)
    )


# ============================================================
# 7. TIE SIZE
# ============================================================

def get_tie_size(row):

    labels = row.dropna().tolist()

    if len(labels) == 0:
        return 0

    counts = Counter(labels)

    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # Number of tied labels
    return len(winners) if len(winners) > 1 else 0


# ============================================================
# 8. CREATE GROUND-TRUTH COLUMNS
# ============================================================

df["ground_truth_label"] = (
    df[reviewer_cols]
    .apply(
        majority_vote_severity,
        axis=1
    )
)

df["ground_truth_votes"] = (
    df[reviewer_cols]
    .apply(
        majority_vote_count,
        axis=1
    )
)

df["ground_truth_agreement"] = (
    df["ground_truth_votes"]
    / len(reviewer_cols)
)


# ============================================================
# 9. CREATE ORIGINAL TIE INFORMATION
# ============================================================

df["tie_labels"] = (
    df[reviewer_cols]
    .apply(
        get_tie_labels,
        axis=1
    )
)

df["tie_size"] = (
    df[reviewer_cols]
    .apply(
        get_tie_size,
        axis=1
    )
)

# Indicates that this tweet originally had a tie
df["tie_resolved_by_severity"] = (
    df["tie_size"] > 0
)


# ============================================================
# 10. SEPARATE TIE CASES
# ============================================================

df_ties = df[
    df["tie_size"] > 0
].copy()


# ============================================================
# 11. FILE NAMES
#
# Keep the original filenames.
# ============================================================

ground_truth_file = (
    "Veteran_Tweet_Label_PTSD_majority_vote_ground_truth.csv"
)

tie_file = (
    "Veteran_Tweet_Label_PTSD_majority_vote_ground_truth_ties.csv"
)


# ============================================================
# 12. SAVE FILES
# ============================================================

# All tweets, including severity-resolved ties
df.to_csv(
    ground_truth_file,
    index=False
)

# Only the original tie cases
df_ties.to_csv(
    tie_file,
    index=False
)


# ============================================================
# 13. GROUND-TRUTH STATISTICS
# ============================================================

total_tweets = len(df)

total_ties = len(df_ties)

total_clear_majority = (
    total_tweets - total_ties
)

tie_percentage = (
    total_ties / total_tweets * 100
)

clear_majority_percentage = (
    total_clear_majority / total_tweets * 100
)


print("\n==========================================")
print("GROUND-TRUTH STATISTICS")
print("==========================================")

print(
    f"Total tweets: "
    f"{total_tweets}"
)

print(
    f"Tweets with clear majority: "
    f"{total_clear_majority} "
    f"({clear_majority_percentage:.2f}%)"
)

print(
    f"Tweets with tie: "
    f"{total_ties} "
    f"({tie_percentage:.2f}%)"
)


# ============================================================
# 14. TIE CASE STATISTICS
# ============================================================

print("\n==========================================")
print("TIE CASE STATISTICS")
print("==========================================")

print(
    f"Total tie cases: "
    f"{total_ties}"
)

print(
    f"Percentage of tweets with ties: "
    f"{tie_percentage:.2f}%"
)


# ============================================================
# 15. TIE-SIZE DISTRIBUTION
# ============================================================

print("\nTie-size distribution:")

if total_ties > 0:

    tie_distribution = (
        df_ties["tie_size"]
        .value_counts()
        .sort_index()
    )

    for size, count in tie_distribution.items():

        percentage = (
            count / total_ties * 100
        )

        print(
            f"{size}-way tie: "
            f"{count} tweets "
            f"({percentage:.2f}%)"
        )

else:

    print("No tie cases found.")


# ============================================================
# 16. TIED LABEL COMBINATIONS
# ============================================================

print("\n==========================================")
print("TIED LABEL COMBINATIONS")
print("==========================================")

if total_ties > 0:

    tie_combinations = (
        df_ties["tie_labels"]
        .value_counts()
    )

    print(tie_combinations)

else:

    print("No tie combinations found.")


# ============================================================
# 17. SEVERITY-BASED TIE RESOLUTION
# ============================================================

print("\n==========================================")
print("SEVERITY-BASED TIE RESOLUTION")
print("==========================================")

resolved_ties = df_ties[
    df_ties["ground_truth_label"].notna()
].copy()

if len(resolved_ties) > 0:

    print(
        f"Ties resolved by severity: "
        f"{len(resolved_ties)}"
    )

    print("\nTie resolution examples:")

    print(
        resolved_ties[
            [
                "tie_labels",
                "ground_truth_label",
                "ground_truth_votes"
            ]
        ].head(10)
    )

else:

    print(
        "No ties were resolved by severity."
    )


# ============================================================
# 18. FINAL GROUND-TRUTH LABEL DISTRIBUTION
# ============================================================

LABEL_ORDER = [
    "1-No Concern",
    "2- Distressed Self-Disclosure",
    "3-Self-Identified Crisis Symptoms and Seeking Support",
    "4- Self-Identified Crisis Symptoms but Not Seeking Support",
    "5- Acute Crisis with Suicidal Ideation",
    "6- Not Relevant to the Research",
    "Other"
]

label_distribution = (
    df["ground_truth_label"]
    .value_counts()
    .reindex(
        LABEL_ORDER,
        fill_value=0
    )
)

label_percentage = (
    label_distribution
    / total_tweets
    * 100
)

distribution = pd.DataFrame({
    "Count": label_distribution,
    "Percentage": label_percentage.round(2)
})


print("\n==========================================")
print("FINAL GROUND-TRUTH LABEL DISTRIBUTION")
print("==========================================")

print(distribution)


# ============================================================
# 19. FILES CREATED
# ============================================================

print("\n==========================================")
print("FILES CREATED")
print("==========================================")

print(
    f"Ground truth: {ground_truth_file}"
)

print(
    f"Tie cases:    {tie_file}"
)

Number of reviewers: 8
Number of tweets: 149

GROUND-TRUTH STATISTICS
Total tweets: 149
Tweets with clear majority: 140 (93.96%)
Tweets with tie: 9 (6.04%)

TIE CASE STATISTICS
Total tie cases: 9
Percentage of tweets with ties: 6.04%

Tie-size distribution:
2-way tie: 7 tweets (77.78%)
3-way tie: 2 tweets (22.22%)

TIED LABEL COMBINATIONS
tie_labels
2- Distressed Self-Disclosure (2 votes) | 5- Acute Crisis with Suicidal Ideation (2 votes)                                                                      2
1-No Concern (3 votes) | 2- Distressed Self-Disclosure (3 votes)                                                                                                1
1-No Concern (3 votes) | 3-Self-Identified Crisis Symptoms and Seeking Support (3 votes)                                                                        1
1-No Concern (3 votes) | 6- Not Relevant to the Research (3 votes)                                                                                              1


In [11]:
# ============================================================
# Print tied tweets with reviewer labels
# ============================================================

def is_tie(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return False

    counts = Counter(labels)
    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    return len(winners) > 1


tie_mask = df[reviewer_cols].apply(
    is_tie,
    axis=1
)

tied_tweets = df.loc[
    tie_mask,
    ["tweet", *reviewer_cols, "ground_truth_label",
     "ground_truth_votes", "ground_truth_agreement"]
].copy()

print("\n==========================================")
print("TIED TWEETS")
print("==========================================")

print(f"Number of tied tweets: {len(tied_tweets)}")

for idx, row in tied_tweets.iterrows():

    print("\n------------------------------------------")
    print(f"Tweet index: {idx}")
    print(f"Tweet: {row['tweet']}")

    print("\nReviewer labels:")

    for reviewer in reviewer_cols:
        print(
            f"  {reviewer}: "
            f"{row[reviewer]}"
        )

    print(
        f"\nGround-truth label: "
        f"{row['ground_truth_label']}"
    )

    print(
        f"Majority votes: "
        f"{row['ground_truth_votes']}"
    )

    print(
        f"Agreement: "
        f"{row['ground_truth_agreement']:.2f}"
    )


TIED TWEETS
Number of tied tweets: 9

------------------------------------------
Tweet index: 3
Tweet: Round&round electricity goes where it stop PLEASE MAKE IT COME BACK ON11

Reviewer labels:
  reviewer1: 2- Distressed Self-Disclosure
  reviewer2: 1-No Concern
  reviewer3: 1-No Concern
  reviewer4: 4- Self-Identified Crisis Symptoms but Not Seeking Support
  reviewer5: 2- Distressed Self-Disclosure
  reviewer6: 1-No Concern
  reviewer7: 2- Distressed Self-Disclosure
  reviewer8: 6- Not Relevant to the Research

Ground-truth label: 2- Distressed Self-Disclosure
Majority votes: 3
Agreement: 0.38

------------------------------------------
Tweet index: 5
Tweet: I’m done #bipolar #bipolarclub113

Reviewer labels:
  reviewer1: Other
  reviewer2: 2- Distressed Self-Disclosure
  reviewer3: 2- Distressed Self-Disclosure
  reviewer4: 6- Not Relevant to the Research
  reviewer5: 5- Acute Crisis with Suicidal Ideation
  reviewer6: 5- Acute Crisis with Suicidal Ideation
  reviewer7: 4- Self-Ide

In [13]:
# ============================================================
# Print tied tweets with label vote counts
# ============================================================

print("\n==========================================")
print("TIED TWEETS WITH LABEL COUNTS")
print("==========================================")

for idx, row in df.loc[tie_mask].iterrows():

    labels = row[reviewer_cols].dropna().tolist()
    counts = Counter(labels)

    max_count = max(counts.values())

    tied_labels = {
        label: count
        for label, count in counts.items()
        if count == max_count
    }

    print("\n------------------------------------------")
    print(f"Tweet index: {idx}")
    print(f"Tweet: {row['tweet']}")

    print("\nTied labels:")

    for label, count in tied_labels.items():
        print(f"  {label}: {count} votes")

    print("\nAll reviewer labels:")

    for reviewer in reviewer_cols:
        print(f"  {reviewer}: {row[reviewer]}")

    print(
        f"\nGround-truth label: "
        f"{row['ground_truth_label']}"
    )


TIED TWEETS WITH LABEL COUNTS

------------------------------------------
Tweet index: 3
Tweet: Round&round electricity goes where it stop PLEASE MAKE IT COME BACK ON11

Tied labels:
  2- Distressed Self-Disclosure: 3 votes
  1-No Concern: 3 votes

All reviewer labels:
  reviewer1: 2- Distressed Self-Disclosure
  reviewer2: 1-No Concern
  reviewer3: 1-No Concern
  reviewer4: 4- Self-Identified Crisis Symptoms but Not Seeking Support
  reviewer5: 2- Distressed Self-Disclosure
  reviewer6: 1-No Concern
  reviewer7: 2- Distressed Self-Disclosure
  reviewer8: 6- Not Relevant to the Research

Ground-truth label: 2- Distressed Self-Disclosure

------------------------------------------
Tweet index: 5
Tweet: I’m done #bipolar #bipolarclub113

Tied labels:
  2- Distressed Self-Disclosure: 2 votes
  5- Acute Crisis with Suicidal Ideation: 2 votes

All reviewer labels:
  reviewer1: Other
  reviewer2: 2- Distressed Self-Disclosure
  reviewer3: 2- Distressed Self-Disclosure
  reviewer4: 6- Not Re

In [12]:
import pandas as pd
from collections import Counter

# ============================================================
# Read merged data
# ============================================================

df = pd.read_csv(
    "Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged.csv"
)

# ============================================================
# Identify reviewer columns
# ============================================================

reviewer_cols = [
    c for c in df.columns
    if c.startswith("reviewer")
]

# ============================================================
# Majority vote
# ============================================================

def majority_vote(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return None

    counts = Counter(labels)
    max_count = max(counts.values())

    winners = [
        label
        for label, count in counts.items()
        if count == max_count
    ]

    # Clear majority
    if len(winners) == 1:
        return winners[0]

    # Tie
    return " | ".join(sorted(winners))


# ============================================================
# Number of votes for majority label
# ============================================================

def majority_vote_count(row):
    labels = row.dropna().tolist()

    if len(labels) == 0:
        return 0

    return max(Counter(labels).values())


# ============================================================
# Create ground-truth columns
# ============================================================

df["ground_truth_label"] = (
    df[reviewer_cols]
    .apply(majority_vote, axis=1)
)

df["ground_truth_votes"] = (
    df[reviewer_cols]
    .apply(majority_vote_count, axis=1)
)

df["ground_truth_agreement"] = (
    df["ground_truth_votes"] / len(reviewer_cols)
)

# ============================================================
# Save
# ============================================================

output_file = (
    "Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged_ground_truth.csv"
)

df.to_csv(output_file, index=False)

# ============================================================
# Results
# ============================================================

print("Ground-truth file created:")
print(output_file)

print("\nGround-truth label distribution:")
print(df["ground_truth_label"].value_counts())

print("\nMajority vote distribution:")
print(df["ground_truth_votes"].value_counts().sort_index())

FileNotFoundError: [Errno 2] No such file or directory: 'Veteran_Tweet_Label_PTSD_Other_Not_Relevant_merged.csv'